In [ ]:
# ============================================
# Task 3: Feature Engineering - Event Impact Model
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
import os


# ============================================
# Load Dataset (CSV)
# ============================================

data_file = "../data/raw/ethiopia_fi_unified_data.csv"
impact_file = "../data/processed/new_impact_links.csv"


data = pd.read_csv(data_file)
impact = pd.read_csv(impact_file)


print("="*60)
print("DATA LOADED")
print("="*60)

print("Data shape:", data.shape)
print("Impact shape:", impact.shape)



# ============================================
# Convert Dates
# ============================================

data["observation_date"] = pd.to_datetime(
    data["observation_date"],
    errors="coerce"
)



# ============================================
# Separate Events and Observations
# ============================================

events = data[
    data["record_type"].str.lower() == "event"
].copy()


observations = data[
    data["record_type"].str.lower() == "observation"
].copy()



# Create event date

events["event_date"] = pd.to_datetime(
    events["observation_date"],
    errors="coerce"
)



print("\nEvents:", len(events))
print("Observations:", len(observations))



# ============================================
# Merge Impact Links with Events
# ============================================

event_info = events[
    [
        "record_id",
        "category",
        "pillar",
        "observation_date",
        "event_date"
    ]
]


impact_df = impact.merge(
    event_info,
    left_on="parent_id",
    right_on="record_id",
    how="left"
)



print("\nMerged impact-event shape:")
print(impact_df.shape)



# ============================================
# Event Summary
# ============================================

print("\n")
print("="*60)
print("EVENT SUMMARY")
print("="*60)


print(
    impact_df[
        [
            "parent_id",
            "category",
            "related_indicator",
            "impact_direction",
            "impact_magnitude",
            "lag_months"
        ]
    ]
)



# ============================================
# Event Indicator Association Matrix
# ============================================

association = impact_df.pivot_table(
    index="category",
    columns="related_indicator",
    values="impact_magnitude",
    aggfunc="mean"
)



print("\nAssociation Matrix")
print(association)



plt.figure(figsize=(12,6))

sns.heatmap(
    association,
    annot=True,
    cmap="RdYlGn",
    center=0
)

plt.title(
    "Event-Indicator Association Matrix"
)

plt.tight_layout()
plt.show()



# ============================================
# Event Impact Function
# ============================================

def event_effect(
    obs_date,
    event_date,
    magnitude,
    lag
):

    if pd.isna(event_date):
        return 0


    activation_date = (
        event_date 
        + pd.DateOffset(
            months=int(lag)
        )
    )


    if obs_date >= activation_date:
        return magnitude

    return 0



# ============================================
# Generate Predictions
# ============================================

results = []



for indicator in impact_df[
    "related_indicator"
].dropna().unique():


    obs = observations[
        observations["indicator_code"] == indicator
    ].copy()



    if obs.empty:
        continue



    obs = obs.sort_values(
        "observation_date"
    )



    links = impact_df[
        impact_df["related_indicator"] == indicator
    ]



    predictions = []



    for obs_date in obs["observation_date"]:


        total_effect = 0



        for _, link in links.iterrows():


            effect = event_effect(
                obs_date,
                link["event_date"],
                link["impact_magnitude"],
                link["lag_months"]
            )



            if str(
                link["impact_direction"]
            ).lower() == "negative":

                effect = -abs(effect)



            total_effect += effect



        predictions.append(
            total_effect
        )



    obs["predicted_effect"] = predictions

    obs["indicator"] = indicator


    results.append(obs)




# ============================================
# Validation
# ============================================

validation = []


if len(results) > 0:


    results = pd.concat(
        results,
        ignore_index=True
    )


    print("\n")
    print("="*60)
    print("VALIDATION RESULTS")
    print("="*60)



    for indicator in results["indicator"].unique():


        temp = results[
            results["indicator"] == indicator
        ]



        mae = mean_absolute_error(
            temp["value_numeric"],
            temp["predicted_effect"]
        )


        validation.append(
            [
                indicator,
                mae
            ]
        )



        plt.figure(figsize=(8,4))


        plt.plot(
            temp["observation_date"],
            temp["value_numeric"],
            marker="o",
            label="Observed"
        )


        plt.plot(
            temp["observation_date"],
            temp["predicted_effect"],
            marker="s",
            label="Predicted Event Effect"
        )



        plt.title(indicator)

        plt.xlabel(
            "Date"
        )

        plt.ylabel(
            "Value"
        )

        plt.legend()

        plt.grid(True)

        plt.show()



    validation = pd.DataFrame(
        validation,
        columns=[
            "Indicator",
            "MAE"
        ]
    )



    print(validation)



else:

    print(
        "No matching indicators found."
    )




# ============================================
# Save Results
# ============================================

os.makedirs(
    "../reports",
    exist_ok=True
)



association.to_csv(
    "../reports/event_indicator_association_matrix.csv"
)



if len(results) > 0:

    validation.to_csv(
        "../reports/impact_validation.csv",
        index=False
    )



print("\n")
print("="*60)
print("FILES SAVED")
print("="*60)

print(
    "event_indicator_association_matrix.csv"
)


if len(results) > 0:
    print(
        "impact_validation.csv"
    )



# ============================================
# Model Assumptions
# ============================================

print("\nMODEL ASSUMPTIONS")

print("- Effects start after lag_months.")
print("- Event effects are additive.")
print("- Impact magnitude remains constant.")
print("- Positive and negative impacts are separated.")
print("- No event decay is applied.")
print("- External factors are not included.")